# 📄 DOCUMENT VQA LIVE DEMO (QWEN2-VL-2B + LORA) TRÊN GPU TESLA T4
Hệ thống hỏi đáp và trích xuất thông tin hóa đơn tiếng Việt chạy hoàn toàn trên GPU Nvidia Tesla T4.

In [ ]:
# 1. Gỡ xung đột torchao và cài đặt các thư viện chuẩn
!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers==4.46.2" "peft==0.13.2" "accelerate==0.34.2" "gradio>=4.0.0" pillow torchvision

import sys
for mod in list(sys.modules.keys()):
    if any(mod.startswith(k) for k in ["transformers", "peft", "accelerate", "torchao", "qwen_vl_utils"]):
        del sys.modules[mod]

import os
import time
import torch
from PIL import Image
import gradio as gr
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel

if torch.cuda.is_available():
    print(f"🔥 Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    print(f"🧠 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Cảnh báo: Chưa bật GPU! Hãy vào Settings -> Accelerator -> Chọn GPU T4 x1.")


In [ ]:
# 2. Tìm kiếm và nạp LoRA Adapter
adapter_dir = None
search_paths = [
    "/kaggle/input/docvqa-lora-adapters",
    "/kaggle/input/docvqa-lora-adapters/lora_adapters",
    "/kaggle/input"
]

for sp in search_paths:
    if os.path.exists(sp):
        for root, dirs, files in os.walk(sp):
            if "adapter_config.json" in files:
                adapter_dir = root
                break
    if adapter_dir:
        break

print(f"📍 LoRA Adapter Path: {adapter_dir}")

# 3. Nạp Base Model & LoRA weights vào GPU
model_name = "Qwen/Qwen2-VL-2B-Instruct"
print(f"⏳ Đang nạp Processor từ {model_name}...")
processor = AutoProcessor.from_pretrained(model_name, min_pixels=256*28*28, max_pixels=1280*28*28)

print("⏳ Đang nạp Base Model vào VRAM (FP16)... ")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if adapter_dir and os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
    print(f"🚀 Đang gắn LoRA Adapter từ {adapter_dir}...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    print("🎉 NẠP THÀNH CÔNG FINE-TUNED MODEL (QWEN2-VL + LORA)!")
else:
    print("ℹ️ Chạy trực tiếp Base Model (hoặc chưa gắn dataset adapter).")
    model = base_model.eval()


In [ ]:
# 4. Hàm suy luận VQA (Inference function)
def predict_vqa(image, question):
    if image is None:
        return "⚠️ Vui lòng tải lên ảnh hóa đơn hoặc chứng từ."
    if not question or not question.strip():
        question = "Trích xuất các trường thông tin: Tên người bán, Mã số thuế, Ngày lập, Tổng tiền thanh toán."
    
    t0 = time.time()
    try:
        image_rgb = image.convert("RGB")
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_rgb},
                    {"type": "text", "text": question.strip()}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False
            )
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            response = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0].strip()
            
        latency = time.time() - t0
        device_label = "GPU Tesla T4" if torch.cuda.is_available() else "CPU"
        return f"{response}\n\n⏱️ Thời gian xử lý: {latency:.2f} giây ({device_label})"
    except Exception as exc:
        return f"❌ Lỗi xử lý: {str(exc)}"


In [ ]:
# 5. Xây dựng giao diện Gradio và khởi chạy
with gr.Blocks(title="Document VQA - Qwen2-VL-2B (GPU Tesla T4)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📄 Hệ Thống Document Visual Question Answering (DocVQA)")
    gr.Markdown("💡 Trợ lý AI hỏi đáp và bóc tách thông tin hóa đơn tiếng Việt chạy trực tiếp trên **GPU NVIDIA Tesla T4**.")
    
    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Tải lên ảnh Hóa đơn / Chứng từ")
            q_input = gr.Textbox(
                lines=2,
                placeholder="Ví dụ: Tổng tiền thanh toán trên hóa đơn là bao nhiêu?",
                label="Câu hỏi hoặc Yêu cầu trích xuất"
            )
            btn_submit = gr.Button("🚀 Phân tích & Trả lời", variant="primary")
            
            gr.Examples(
                examples=[
                    ["Tổng tiền thanh toán trên hóa đơn là bao nhiêu?"],
                    ["Tên cửa hàng / bên bán trên hóa đơn là gì?"],
                    ["Mã số thuế của bên bán là gì?"],
                    ["Ngày giờ lập hóa đơn là khi nào?"],
                    ["Trích xuất toàn bộ thông tin dưới dạng JSON."]
                ],
                inputs=[q_input],
                label="💡 Gợi ý câu hỏi mẫu"
            )
            
        with gr.Column(scale=1):
            txt_output = gr.Textbox(lines=12, label="Kết quả phản hồi từ AI (Qwen2-VL + LoRA)")
            
    btn_submit.click(fn=predict_vqa, inputs=[img_input, q_input], outputs=txt_output)

print("🌐 Đang mở Server Gradio...")
try:
    demo.queue().launch(share=True, debug=True)
except Exception as e:
    print(f"⚠️ Không thể tạo link Public Gradio: {e}. Đang mở chế độ nhúng Inline...")
    demo.queue().launch(share=False, debug=True, inline=True)
